In [180]:
import os
from dotenv import load_dotenv
from typing import List, Literal, Optional, TypedDict, Dict
from langgraph.graph import END, StateGraph, state
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
import pandas as pd
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
import re
import subprocess
from openai import OpenAI
import datetime

In [181]:
if not os.path.exists("./generatedFiles"):
    os.makedirs("./generatedFiles")
tempPath = f"./generatedFiles/Files_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
if not os.path.exists(tempPath):
    os.makedirs(tempPath)

In [182]:
load_dotenv()
envVars = pd.Series(os.environ)
envVars = envVars[["OPENAI_API_KEY", "AGENT_MODEL", "AGENT_MAX_RETRIES"]]
for i in envVars.index: print(i, ": ", envVars.loc[i][:8], sep = "")

OPENAI_API_KEY: <redacted>
AGENT_MODEL: gpt-4o
AGENT_MAX_RETRIES: 16


In [183]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("=== 你当前可用的 GPT 模型系列 ===")
models = client.models.list()
for m in models.data:
    # 过滤一下，只看核心的 gpt 模型
    if "gpt" in m.id:
        print(m.id)

=== 你当前可用的 GPT 模型系列 ===
gpt-3.5-turbo
gpt-3.5-turbo-16k
gpt-4-0613
gpt-4
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-1106
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
gpt-4o-mini-tts
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-audio-2025-08-28
gpt-realtime
gpt-realtime-2025-08-28
gpt-audio
gpt-5-codex
gpt-image-1-mini
gpt-5-pro-2025-10-06
gpt-5-pro
gpt-audio-mini
gpt-audio-mini-2025-10-06
gpt-5-search-api
gpt-realtime-mini
gpt-realtime-mini-2025-10-06
gpt-5-search-api-2025-10-14
gpt-5.1-chat-latest
gpt-5.1-2025-11-13
gpt-5.1
gpt-5.1-codex
gpt-5.1-codex-min

In [184]:
set_llm_cache(InMemoryCache())

In [185]:
def _llm() -> ChatOpenAI:
    model = os.environ.get("AGENT_MODEL", "gpt-4o-mini")
    # Low temperature — we want reproducible, focused code.
    return ChatOpenAI(model = model, temperature = 0.2)

llm = _llm()

In [186]:
task = {
    "name": "column_maximim_strategy",
    "problem": "Given a numpy matrix A, find a probability vector as another column numpy matrix p, such that there exist no probability vector q satisfies min(A@q) > min(A@p)",
    "signature": "def column_maximin(A: numpy.matrix) -> numpy.matrix:",
    "public_tests": [
      "assert bool(min(np.matrix([[-54, 32],[62, -37],[-13, 7]])@column_maximin(np.matrix([[-54, 32],[62, -37],[-13, 7]])))[0, 0] >= -0.395)"
    ],
    "hidden_tests": [
      "assert bool(min(np.matrix([[-65, 39],[62, 40],[-3, -11]])@column_maximin(np.matrix([[-65, 39],[62, 40],[-3, -11]])))[0, 0] >= -7.43)"
    ]
}

In [187]:
class TestResult(TypedDict):
    """Outcome of running the generated code against the task's tests."""
    passed: bool
    stdout: str
    stderr: str
    failing_test: Optional[str]

Workflow = Literal["Init", "Specify", "Generate", "Test", "Repair", "Done", "Fail", "Plan"]

In [188]:
class AgentState(TypedDict, total = False):
    # --- task description (immutable after Init) -------------------------
    problem: str            # natural-language problem statement
    signature: str          # required function signature, e.g. "def foo(x):"
    public_tests: List[str] # python `assert` statements
    hidden_tests: List[str] # extra tests only used to grade final solution

    # --- mutable during the run ----------------------------------------
    workflow: Workflow      # current control-flow state  (TLA+: pc)
    code: str               # latest generated solution    (TLA+: code)
    last_result: Optional[TestResult]   # last Test outcome (TLA+: tested)
    retries: int            # number of repair attempts so far (TLA+: retries)
    max_retries: int        # upper bound on retries           (TLA+: MaxRetries)

    
    # --- auditing ------------------------------------------------------
    history: List[str]      # human-readable trace for the report

    # PlusCal proof stage:
    approach: str
    subtasks: List[str]
    plusCalSpec: List[str]
    unprovedSubtasks: List[str]
    provedSubtasks: Dict[str, str]
    failedSubtasks: Dict[str, str]  # <--- 新增：存放达到最大重试次数的子任务及其最后代码
    current_task_desc: str  # 当前正在处理的子任务描述
    current_tla: str        # 当前生成的 TLA/PlusCal 代码
    current_cfg: str        # 当前的配置文件
    current_error: str      # 当前遇到的编译/校验错误

In [189]:
currentState: AgentState = {
    "problem": task["problem"],
    "signature": task["signature"],
    "public_tests": task["public_tests"],
    "hidden_tests": task.get("hidden_tests", []),
    "workflow": "Init",
    "code": "",
    "plusCalSpec": [],
    "last_result": None,
    "retries": 0,
    "max_retries": 8,
    "history": [],
    "approach": "",
    "subtasks": [],
    "unprovedSubtasks": [],
    "provedSubtasks": {},
    "failedSubtasks": {},
}

In [190]:
def extract_tla_and_cfg(llm_output: str) -> tuple[str, str, str]:
    tla_match = re.search(r"```tla\s+(.*?)\s+```", llm_output, re.DOTALL)
    tla_code = tla_match.group(1).strip() if tla_match else ""

    cfg_match = re.search(r"```cfg\s+(.*?)\s+```", llm_output, re.DOTALL)
    cfg_code = cfg_match.group(1).strip() if cfg_match else ""

    module_match = re.search(r"----\s*MODULE\s+(\w+)\s*----", tla_code)
    module_name = module_match.group(1) if module_match else "TempModule"
    return module_name, tla_code, cfg_code

In [191]:
def init_action(state: AgentState) -> AgentState:
    """Init: set the workflow counters. Mirrors TLA+ ``Init``."""
    return {
        "workflow": "Plan",
        "retries": 0,
        "max_retries": state.get("max_retries", int(os.environ.get("AGENT_MAX_RETRIES", "3"))),
        "history": state.get("history", []) + ["Init -> Plan"],
    }

In [192]:
planSysMessage = (
    "You are a careful programmer and has been given a problem."
    "First analyze the problem to see if it is equivalent to problems with available solutions"
    "If yes then summarize a solution into a plan"
    "If no then suggest a plan"
    "Note that at this stage we are looking for a plan, not an implementation"
    "Try to describe the plan using natural language and PlusCal"
)
def plan_action(state: AgentState) -> AgentState:
    prompt = (
        f"Problem:\n{state['problem']}\n\n"
        f"Required signature:\n{state['signature']}\n\n"
        "Before generating the implementation, try to design an approach to solve this problem"
    )
    resp = llm.invoke([ SystemMessage(content = planSysMessage), 
                        HumanMessage(content = prompt)])
    return {"approach": resp.content, 
            "workflow": "Spec"}

In [193]:
splitSysMessage = (
    "You are a careful programmer and has a plan.\n"
    "Split the plan into independent sub-tasks, "
    "such that each task focus on one component of the plan.\n"
    "If the task is easy then one sub-task is acceptable.\n"
    "CRITICAL RULE: You MUST start every single sub-task with the exact delimiter '====SUBTASK===='.\n"
    "Do not use '====SUBTASK====' anywhere else. Do not output any introductory text before the first sub-task.\n"
    "Note that at this stage we wish to specify the procedure, not to implement.\n"
    "Breakdown the steps and specify them using PlusCal."
)

def split_action(state: AgentState) -> AgentState:
    prompt = (
        f"Problem:\n{state['problem']}\n\n"
        f"Required signature:\n{state['signature']}\n\n"
        f"Approach:\n{state['approach']}"
    )
    resp = llm.invoke([ SystemMessage(content = splitSysMessage), 
                        HumanMessage(content = prompt)])
    
    # 按照专属分隔符拆分
    raw_subtasks = resp.content.split("====SUBTASK====")
    
    # 过滤掉空的或者只有空格的字符串，并排除掉可能存在的极短废话前缀
    subtasks = [task.strip() for task in raw_subtasks if task.strip() and len(task.strip()) > 10]
    
    print(f"[调试信息] 成功切分出 {len(subtasks)} 个子任务") # 加一句打印方便观察
    
    return {"subtasks": subtasks, 
            "unprovedSubtasks": subtasks.copy(),
            "provedSubtasks": {},
            "workflow": "Spec"}

In [194]:
proveSysMessage = (
    "You are a formal verification expert. "
    "We are using the PlusCal algorithm language. Your job is to take a PlusCal sub-task and write the PlusCal code. "
    "CRITICAL REQUIREMENTS:\n"
    "1. You must output TWO code blocks: A TLA+ module containing the PlusCal algorithm inside ```tla ... ``` and a minimal TLC config inside ```cfg ... ```.\n"
    "2. DO NOT write the TLA+ translation (like Init, Next, Spec). Just write the PlusCal algorithm inside the comment block.\n"
    "3. For the cfg block, simply use `SPECIFICATION Spec`."
    "- Uninitialized Input Variables (CRITICAL): TLC must evaluate real values! If your algorithm takes an input matrix or sequence (like `A`), you MUST initialize it with a concrete dummy value in the variables block. Example: `variables A = << <<1, -1>>, <<-1, 1>> >>;`. Do NOT just write `variables A;`.\n"
    "- Sequence vs Set Length: In TLA+, a matrix is a Sequence of Sequences (Tuples). Use `Len(A)` to get the number of rows. DO NOT use `Cardinality(A)` on a Sequence! Cardinality is only for Sets (like `{1, 2}`).\n"
    "- 'print' KEYWORD ERROR: PlusCal `print` takes exactly ONE expression. DO NOT use commas like `print a, b`. If you must print multiple items, pack them into a tuple: `print <<a, b>>;`. Better yet, REMOVE `print` statements entirely, they are not needed for verification.\n"
    "- Undefined Operators (CRITICAL): DO NOT invent magical operators without defining them! If you use a helper function, you MUST either define it mathematically in the TLA+ module above the algorithm block, or declare it as a `CONSTANT` and assign it a dummy value in the CFG block.\n"
)

def prove_action(state: AgentState) -> AgentState:
    """生成 PlusCal 规约（如果有未完成的任务）"""
    unproved = state.get("unprovedSubtasks", []).copy()
    
    # 如果当前没有正在处理的任务，就从队列里弹出一个新的
    current_task = state.get("current_task_desc", "")
    if not current_task and unproved:
        current_task = unproved.pop(0)
        
    prompt = f"Problem context:\n{state.get('approach', '')}\n\nCurrent Sub-task to prove:\n{current_task}"
    resp = llm.invoke([SystemMessage(content=proveSysMessage), HumanMessage(content=prompt)])
    
    module_name, tla_code, cfg_code = extract_tla_and_cfg(resp.content) # 记得确保 extract 函数在这个 Cell 之前已经定义
    
    return {
        "unprovedSubtasks": unproved,
        "current_task_desc": current_task,
        "current_tla": tla_code,
        "current_cfg": cfg_code,
        "current_error": "", # 清空之前的错误
        "history": state.get("history", []) + [f"Prove -> 为子任务生成了代码: {module_name}"]
    }

def route_after_prove(state: AgentState) -> str:
    unproved = state.get("unprovedSubtasks", [])
    if len(unproved) > 0: return "continue_proving"
    else: return "all_proved"

In [195]:
repairSysMessage = (
    "You are a PlusCal/TLA+ debugging expert. "
    "The PlusCal code you just generated failed to compile. "
    "CRITICAL RULE: strictly base your fix on the 'Previous Code'. Only make minimal necessary edits.\n"
    "Here is the Ultimate PlusCal Syntax Guide to fix your errors:\n"
    "- 'return' KEYWORD ERROR (CRITICAL): PlusCal DOES NOT support returning values like `return p;`! The `return;` statement can only be used EMPTY to exit a `procedure`. In the main algorithm body, simply assign your final result to a variable and let the algorithm end. REMOVE any `return <var>;` statements!\n"
    "- 'assume' KEYWORD ERROR: PlusCal DOES NOT have an `assume` statement! Use `assert condition;` to check conditions, or `with var \\in Set do ... end with;` to non-deterministically pick a value.\n"
    "- Assignment Operator: ALWAYS use `:=` for assignment. `=` is ONLY for mathematical equality checking.\n"
    "- Tuple Unpacking: PlusCal DOES NOT support `(a, b) := ...` or `<<a, b>> := ...`. You MUST use parallel assignment with `||` and indices. Example: `p := LP_Solve(A)[1] || t := LP_Solve(A)[2];`\n"
    "- Missing label: Insert labels (e.g., `L1:`) before `while` loops or after macro/procedure calls.\n"
    "- Multiple assignment: Do not assign to the same variable twice in one labeled step.\n"
    "Now, analyze the error log, fix the specific line, and output the ENTIRE corrected TLA+ module (with the (* --algorithm ... *) block) and the CFG block."
    "- Uninitialized Input Variables (CRITICAL): TLC must evaluate real values! If your algorithm takes an input matrix or sequence (like `A`), you MUST initialize it with a concrete dummy value in the variables block. Example: `variables A = << <<1, -1>>, <<-1, 1>> >>;`. Do NOT just write `variables A;`.\n"
    "   - Sequence vs Set Length: In TLA+, a matrix is a Sequence of Sequences (Tuples). Use `Len(A)` to get the number of rows. DO NOT use `Cardinality(A)` on a Sequence! Cardinality is only for Sets (like `{1, 2}`).\n"
    "- 'print' KEYWORD ERROR: PlusCal `print` takes exactly ONE expression. DO NOT use commas like `print a, b`. If you must print multiple items, pack them into a tuple: `print <<a, b>>;`. Better yet, REMOVE `print` statements entirely, they are not needed for verification.\n"
    "- Undefined Operators (CRITICAL): DO NOT invent magical operators without defining them! If you use a helper function, you MUST either define it mathematically in the TLA+ module above the algorithm block, or declare it as a `CONSTANT` and assign it a dummy value in the CFG block.\n"
)

def check_action(state: AgentState) -> AgentState:
    """物理执行 pcal 和 tlc 进行严格校验"""
    tla_code = state.get("current_tla", "")
    cfg_code = state.get("current_cfg", "")
    task_desc = state.get("current_task_desc", "")
    proved = state.get("provedSubtasks", {}).copy()
    
    module_match = re.search(r"----\s*MODULE\s+(\w+)\s*----", tla_code)
    module_name = module_match.group(1) if module_match else "TempModule"
    
    target_dir = tempPath
    os.makedirs(target_dir, exist_ok=True)
    tla_file_path = f"{target_dir}/{module_name}.tla"
    cfg_file_path = f"{target_dir}/{module_name}.cfg"
    
    with open(tla_file_path, "w", encoding="utf-8") as f: f.write(tla_code)
    with open(cfg_file_path, "w", encoding="utf-8") as f: f.write(cfg_code)
        
    error_log = ""
    # 1. 运行 pcal
    pcal_res = subprocess.run(["pcal.bat", f"{module_name}.tla"], cwd=target_dir, capture_output=True, text=True)
    if pcal_res.returncode != 0 or "Error" in pcal_res.stdout or "Unrecoverable error" in pcal_res.stdout:
        error_log = f"PlusCal Parsing Error:\n{pcal_res.stdout}"
    else:
        # 2. 运行 tlc
        tlc_res = subprocess.run(["tlc.bat", f"{module_name}.tla"], cwd=target_dir, capture_output=True, text=True)
        if tlc_res.returncode != 0 or "Error:" in tlc_res.stdout:
            error_log = f"TLC Model Checking Error:\n{tlc_res.stdout}"
            
    if error_log:
        # ================= 新增：把大模型搞不定的报错打印给人类看 =================
        # print(f"\n[调试信息] 模块 {module_name} 遇到了错误：\n{error_log[:]}...\n")
        # =========================================================================
        return {"current_error": error_log, "history": state.get("history", []) + [f"Check -> 发现错误，进入 Repair"]}
    else:
        proved[task_desc] = tla_code


    current_specs = state.get("plusCalSpec", []).copy()
    current_specs.append(tla_code)
    
    return {
        "provedSubtasks": proved,
        "plusCalSpec": current_specs, # <--- 状态更新
        "current_task_desc": "", 
        "current_error": "",
        "retries": 0, 
        "history": state.get("history", []) + [f"Check -> {module_name} 校验完美通过！"]
    }

def repair_action(state: AgentState) -> AgentState:
    """如果校验失败，将错误日志喂回给大模型进行修复"""
    prompt = (
        f"Sub-task:\n{state.get('current_task_desc')}\n\n"
        f"Your Previous Code:\n```tla\n{state.get('current_tla')}\n```\n\n"
        f"Compiler/Checker Error Log:\n{state.get('current_error')}\n\n"
        "Please fix the code."
    )
    resp = llm.invoke([SystemMessage(content=repairSysMessage), HumanMessage(content=prompt)])
    module_name, tla_code, cfg_code = extract_tla_and_cfg(resp.content)
    
    return {
        "current_tla": tla_code,
        "current_cfg": cfg_code,
        "retries": state.get("retries", 0) + 1,
        "history": state.get("history", []) + ["Repair -> 尝试了一次修复"]
    }

In [196]:
# 建议在 Cell 18 后面新建一个 Cell 存放这段代码

def skip_action(state: AgentState) -> AgentState:
    """达到最大重试次数，接受失败，记录并清理当前状态"""
    failed = state.get("failedSubtasks", {}).copy()
    task_desc = state.get("current_task_desc", "")
    
    tla_code = state.get("current_tla", "") # 获取最后一次生成的代码
    
    # 记录最后一次失败的代码，供后续人工 Review
    failed[task_desc] = tla_code 
    
    # --- 新增：将失败的妥协代码也追加到 plusCalSpec 列表中 ---
    current_specs = state.get("plusCalSpec", []).copy()
    current_specs.append(tla_code)
    
    return {
        "failedSubtasks": failed,
        "plusCalSpec": current_specs, # <--- 状态更新
        "current_task_desc": "",
        "current_error": "",
        "retries": 0, 
        "history": state.get("history", []) + [f"Skip -> 放弃子任务: 记录失败并继续"]
    }

In [197]:
generateSysMessage = (
    "You are an expert Python developer and algorithm translator. "
    "Your task is to translate a formal PlusCal/TLA+ algorithmic specification into executable Python code. "
    "CRITICAL RULES:\n"
    "1. You must output ONLY valid Python code block starting with ```python and ending with ```.\n"
    "2. Strictly follow the provided function signature.\n"
    "3. Do not include any explanations, usage examples, or tests outside the code block.\n"
    "4. The PlusCal specs provided might contain errors or placeholders (like linear programming solvers). "
    "Use your best judgment to implement these logic components using standard Python libraries (like numpy or scipy.optimize)."
)

def generate_action(state: AgentState) -> AgentState:
    """基于收集到的 PlusCal Spec 列表，汇总并生成最终的 Python 代码"""
    
    # 将收集到的所有阶段的 spec 拼接成一个长字符串作为上下文
    specs_context = "\n\n".join([
        f"--- Stage {i+1} PlusCal Spec ---\n{spec}" 
        for i, spec in enumerate(state.get("plusCalSpec", []))
    ])
    
    prompt = (
        f"Problem Statement:\n{state['problem']}\n\n"
        f"Required Signature:\n{state['signature']}\n\n"
        f"Here is the algorithmic sequence drafted in PlusCal:\n{specs_context}\n\n"
        "Please generate the final Python implementation based on this sequence."
    )
    
    resp = llm.invoke([SystemMessage(content=generateSysMessage), HumanMessage(content=prompt)])
    
    # 提取被 ```python ``` 包裹的代码
    code_match = re.search(r"```python\s+(.*?)\s+```", resp.content, re.DOTALL)
    final_code = code_match.group(1).strip() if code_match else resp.content.strip()
    
    return {
        "workflow": "Generate",
        "code": final_code,
        "history": state.get("history", []) + ["Generate -> 根据汇总的 Spec 生成了最终 Python 代码"]
    }

In [198]:
# --- 提取自同学代码的 System Message ---
# --- 优化后的 Repair System Message ---
REPAIR_PYTHON_SYS = (
    "You are an expert Python debugger and algorithmic engineer. The Python function you generated failed the public tests. "
    "Carefully analyze the traceback and the failing assertions to diagnose the exact issue. "
    "CRITICAL DEBUGGING STRATEGIES:\n"
    "1. API Standard Forms (Critical): If you used optimization, linear algebra, or math libraries (like scipy or numpy), meticulously verify their strictly required standard forms. (e.g., Does the solver minimize or maximize by default? Does it expect <= or >= inequalities?).\n"
    "2. Mathematical Sign Flips: If the error indicates 'unbounded', 'did not converge', or an assertion fails heavily, re-evaluate your mathematical constraint matrix. Sign flips (+/-) and reversed inequality directions are the most common root causes.\n"
    "3. Traceback Root Cause: Do not just patch the surface error. Trace the logic back to the algorithmic blueprint.\n"
    "Output ONLY the fully corrected Python function in a ```python fenced code block. "
    "Do not explain your reasoning, do not output tests."
)

def test_python_action(state: AgentState) -> AgentState:
    """运行 Python 测试用例（沙箱执行）"""
    code = state.get("code", "")
    tests = state.get("public_tests", [])
    
    # 简单的本地代码执行器 (如果同学的代码里有自定义的 run_tests 工具，也可以直接调用)
    passed = True
    error_log = ""
    failing_test = ""
    
    # 将要测试的代码放入独立的命名空间执行
    local_env = {}
    try:
        # 首先尝试编译并执行生成的函数
        # exec(code, {}, local_env)
        exec(code, local_env)
        
        # 逐个运行 public_tests 中的 assert 语句
        for test in tests:
            try:
                # exec(test, globals(), local_env)
                exec(test, local_env)
            except KeyboardInterrupt: raise KeyboardInterrupt
            except Exception as e:
                passed = False
                error_log = str(e)
                failing_test = test
                break

    except KeyboardInterrupt: raise KeyboardInterrupt  
    except Exception as e:
        passed = False
        error_log = str(e)
        failing_test = "Compilation / Setup Error"
        
    result = {
        "passed": passed,
        "stderr": error_log,
        "failing_test": failing_test
    }
    
    return {
        "last_result": result,
        "workflow": "Test",
        "history": state.get("history", []) + [f"Test_Python -> {'Pass' if passed else 'Fail'}"]
    }

def repair_python_action(state: AgentState) -> AgentState:
    """基于错误日志修复 Python 代码"""
    err = state.get("last_result", {})
    
    prompt = (
        f"Problem:\n{state['problem']}\n\n"
        f"Required signature:\n{state['signature']}\n\n"
        f"Previous (failing) code:\n```python\n{state['code']}\n```\n\n"
        f"Failing Test:\n{err.get('failing_test', '')}\n\n"
        f"Error Output:\n{err.get('stderr','')}\n\n"
        "Produce a corrected implementation."
    )
    
    resp = llm.invoke([SystemMessage(content=REPAIR_PYTHON_SYS), HumanMessage(content=prompt)])
    
    # 提取被 ```python ``` 包裹的代码
    code_match = re.search(r"```python\s+(.*?)\s+```", resp.content, re.DOTALL)
    final_code = code_match.group(1).strip() if code_match else resp.content.strip()
    
    return {
        "code": final_code,
        # 可以复用之前的 retries 计数器，也可以在 state 里新建一个 python_retries
        "retries": state.get("retries", 0) + 1, 
        "workflow": "Generate",
        "history": state.get("history", []) + [f"Repair_Python -> 尝试修复代码"]
    }

In [199]:
# 更新路由函数，把 "all_done" 的含义从结束改为去生成代码
def route_after_check(state: AgentState) -> str:
    if state.get("current_error"):
        if state.get("retries", 0) >= state.get("max_retries", 3):
            return "skip"  
        return "repair" 
    else:
        unproved = state.get("unprovedSubtasks", [])
        if len(unproved) > 0:
            return "prove_next" 
        else:
            return "ready_to_generate" # <--- 修改：从 all_done 改为 ready_to_generate

def route_after_skip(state: AgentState) -> str:
    unproved = state.get("unprovedSubtasks", [])
    if len(unproved) > 0:
        return "prove_next"
    else:
        return "ready_to_generate" # <--- 修改：从 all_done 改为 ready_to_generate

In [200]:
def route_after_python_test(state: AgentState) -> str:
    """判断 Python 测试是否通过，或是否达到最大重试次数"""
    result = state.get("last_result")
    if result and result.get("passed"):
        return "done"
    
    # 设定 Python 修复的最大允许次数 (比如 3 次)
    if state.get("retries", 0) < 3: 
        return "repair_python"
        
    return "fail"

# ==========================================
# 确保在这个 Cell 之前，你已经运行了：
# 1. generate_action 的定义 (原 Cell 131)
# 2. test_python_action 和 repair_python_action 的定义 (原 Cell 133)
# ==========================================

def route_after_python_test(state: AgentState) -> str:
    """判断 Python 测试是否通过，或是否达到最大重试次数"""
    result = state.get("last_result")
    if result and result.get("passed"):
        return "done"
    
    # 设定 Python 修复的最大允许次数 (这里设定为 3 次)
    if state.get("retries", 0) < 3: 
        return "repair_python"
        
    return "fail"

In [201]:
def newAgent() -> state.CompiledStateGraph:
    graph = StateGraph(AgentState)
    
    # --- 前半段：PlusCal 规约阶段 ---
    graph.add_node("init", init_action)
    graph.add_node("plan", plan_action)
    graph.add_node("split", split_action)
    graph.add_node("prove", prove_action)
    graph.add_node("check", check_action)
    graph.add_node("repair", repair_action)
    graph.add_node("skip", skip_action)
    
    # --- 后半段：Python 生成与测试阶段 ---
    graph.add_node("generate", generate_action)
    graph.add_node("test_python", test_python_action)
    graph.add_node("repair_python", repair_python_action)

    # --- 1. 前半段连接逻辑 ---
    graph.set_entry_point("init")
    graph.add_edge("init", "plan")
    graph.add_edge("plan", "split")
    graph.add_edge("split", "prove")
    
    graph.add_edge("prove", "check")
    graph.add_edge("repair", "check")

    # Check 和 Skip 后指向 generate
    graph.add_conditional_edges("check", route_after_check, 
        {"repair": "repair", "prove_next": "prove", "skip": "skip", "ready_to_generate": "generate"})
    graph.add_conditional_edges("skip", route_after_skip, 
        {"prove_next": "prove", "ready_to_generate": "generate"})
    
    # --- 2. 后半段拼接逻辑 ---
    # 生成后直接去测试
    graph.add_edge("generate", "test_python")
    
    # 测试后进行路由判断
    graph.add_conditional_edges(
        "test_python",
        route_after_python_test,
        {
            "done": END,               # 通过测试，结束
            "repair_python": "repair_python", # 失败且有余量，去修复
            "fail": END                # 超过重试次数，结束
        }
    )
    
    # 修复后回到测试环节验证
    graph.add_edge("repair_python", "test_python")

    return graph.compile()

# 实例化新的完整 Agent
app = newAgent()

In [202]:
# 1. 实例化编译好的 Agent Graph
app = newAgent()

# 2. 运行 Agent，传入初始状态 (这可能会花费几分钟，具体取决于 API 响应和重试次数)
print("🚀 开始运行 Agent，尝试生成并验证 PlusCal 代码...\n")
final_state = app.invoke(currentState)

# 3. 打印运行历史，查看工作流的流转情况
print("=== 1. 执行轨迹 (Execution History) ===")
for step in final_state.get("history", []):
    print(f" -> {step}")

# 4. 统计并打印成功的子任务
proved:dict = final_state.get("provedSubtasks", {})
print(f"\n=== 2. 成功的子任务 (Proved: {len(proved)}) ===")
for i, (desc, code) in enumerate(proved.items(), 1):
    print(f"  [{i}] {desc[:80]}...")

# 5. 统计并打印触发降级/放弃的子任务
failed:dict = final_state.get("failedSubtasks", {})
print(f"\n=== 3. 放弃的子任务 (Failed/Skipped: {len(failed)}) ===")
for i, (desc, code) in enumerate(failed.items(), 1):
    print(f"  [{i}] {desc[:80]}...")
    
# 如果你想看具体的失败代码，可以取消下面这行的注释
# print(f"\n[最后一次尝试的代码预览]:\n{code[:300]}...\n")

🚀 开始运行 Agent，尝试生成并验证 PlusCal 代码...

[调试信息] 成功切分出 6 个子任务
=== 1. 执行轨迹 (Execution History) ===
 -> Init -> Plan
 -> Prove -> 为子任务生成了代码: MaximinProblem
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Skip -> 放弃子任务: 记录失败并继续
 -> Prove -> 为子任务生成了代码: MaximinProblem
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Check -> 发现错误，进入 Repair
 -> Repair -> 尝试了一次修复
 -> Chec

In [203]:
# 打印所有收集到的代码片段
specs = final_state.get("plusCalSpec", [])
print(f"总共收集到 {len(specs)} 个代码片段。\\n")

for idx, spec_code in enumerate(specs):
    print(f"=== 片段 {idx + 1} ===")
    print(spec_code[:512] + "...\\n")

总共收集到 6 个代码片段。\n
=== 片段 1 ===
---- MODULE MaximinProblem ----
EXTENDS Naturals, Sequences

CONSTANTS m, n

VARIABLES A, p, z

(* --algorithm MaximinProblem -- *)
variables
    A = << <<1, -1>>, <<-1, 1>> >>,
    p = <<0, 0>>,
    z = 0;

begin
    define
        IsProbabilityVector(p) == 
            /\ \A i \in 1..Len(p): p[i] >= 0
            /\ Sum(p) = 1
    end define;

    define
        MinValue(A, p) == Min({A[i] \cdot p : i \in 1..Len(A)})
    end define;

    procedure MaximizeMinValue(A) 
    variable foundP;
    begin
      ...\n
=== 片段 2 ===
---- MODULE MaximinProblem ----
EXTENDS Naturals, Sequences

CONSTANTS A

(* Define a helper function to sum elements of a sequence *)
Sum(seq) == 
    IF seq = <<>> THEN 0
    ELSE Head(seq) + Sum(Tail(seq))

(* --algorithm MaximinLP
variables 
    p = <<0, 0>>,  \* Initialize probability vector with dummy values
    z = 0,         \* Initialize z to represent the minimum value
    result_p;      \* Declare result_p to store the final

In [204]:
print(final_state["code"])

import numpy as np
from scipy.optimize import linprog

def column_maximin(A: np.matrix) -> np.matrix:
    # Number of columns in A
    n = A.shape[1]
    
    # Objective function: maximize z, which is equivalent to minimizing -z
    c = np.zeros(n + 1)
    c[-1] = -1  # We want to maximize z, so minimize -z
    
    # Constraints: A @ p >= z
    # We transform this into A @ p - z <= 0
    A_ub = np.hstack((-A, np.ones((A.shape[0], 1))))
    b_ub = np.zeros(A.shape[0])
    
    # Probability vector constraints: sum(p) = 1 and p >= 0
    A_eq = np.ones((1, n + 1))
    A_eq[0, -1] = 0  # z does not participate in the sum
    b_eq = np.array([1])
    
    # Bounds for each variable: p[i] >= 0 and z is unbounded
    bounds = [(0, 1) for _ in range(n)] + [(None, None)]
    
    # Solve the linear program
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    
    if result.success:
        # Extract the probability vector p
        p = result.